# CET3001 – Practical 01: Exploratory Data Analysis of Student Performance

**Student:** SHRIYASH SAHU  
**Registration Number:** 24BET10001  
**Course:** CET3001 – Educational Data Mining and Learning Analytics  
**Date:** November 2026

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print('All libraries imported successfully.')


## Section A: Dataset Loading and Inspection

In [ ]:
# Load the UCI Student Performance dataset (Mathematics)
df = pd.read_csv('../data/student-mat.csv', sep=';')
print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'
First 5 Records:')
df.head()


In [ ]:
# Last 5 records
print('Last 5 Records:')
df.tail()


In [ ]:
# Data types
print('Column Names and Data Types:')
print(df.dtypes)
print(f"""
Numerical columns ({len(df.select_dtypes(include=[np.number]).columns)}): {df.select_dtypes(include=[np.number]).columns.tolist()}
Categorical columns ({len(df.select_dtypes(include=['object']).columns)}): {df.select_dtypes(include=['object']).columns.tolist()}
""")


In [ ]:
# Missing values and duplicates
print(f'Total missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'
Dataset Info:')
df.info()


## Section B: Data Inspection

In [ ]:
# Target variable summary
print('Target Variable G3 Summary:')
print(df['G3'].describe())
print(f"Students with G3 = 0 (possible dropouts): {(df['G3'] == 0).sum()}")

# Key categorical distributions
for col in ['school', 'sex', 'higher', 'internet']:
    print(f'
{col}:')
    print(df[col].value_counts())


## Section C: Descriptive Statistics

In [ ]:
focus_cols = ['studytime', 'failures', 'absences', 'G1', 'G2', 'G3']
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

desc = df[num_cols].describe().T
desc['median'] = df[num_cols].median()
desc['mode'] = df[num_cols].mode().iloc[0]
desc['variance'] = df[num_cols].var()
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurtosis()
desc['IQR'] = desc['75%'] - desc['25%']
desc


In [ ]:
# Detailed stats for key variables
for col in focus_cols:
    data = df[col]
    print(f"
{col}:")
    print(f"  Mean = {data.mean():.4f}, Median = {data.median():.4f}, Mode = {data.mode().iloc[0]}")
    print(f"  Variance = {data.var():.4f}, Std Dev = {data.std():.4f}")
    print(f"  Min = {data.min()}, Max = {data.max()}, Range = {data.max()-data.min()}")
    print(f"  Q1 = {data.quantile(0.25)}, Q3 = {data.quantile(0.75)}, IQR = {data.quantile(0.75)-data.quantile(0.25)}")
    print(f"  Skewness = {data.skew():.4f}, Kurtosis = {data.kurtosis():.4f}")


## Section D: Visualizations

In [ ]:
# D1: Histogram of G3
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df['G3'], bins=21, range=(-0.5, 20.5), edgecolor='black', color='#4C72B0', alpha=0.85)
ax.set_xlabel('Final Grade (G3)', fontsize=13)
ax.set_ylabel('Number of Students', fontsize=13)
ax.set_title('Figure 1: Distribution of Final Grades (G3)', fontsize=14, fontweight='bold')
ax.set_xticks(range(0, 21))
ax.axvline(df['G3'].mean(), color='red', linestyle='--', lw=2, label=f"Mean = {df['G3'].mean():.2f}")
ax.axvline(df['G3'].median(), color='green', linestyle='-.', lw=2, label=f"Median = {df['G3'].median():.2f}")
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# D2: Box Plot of G3
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(df['G3'], vert=False, patch_artist=True,
           boxprops=dict(facecolor='#4C72B0', alpha=0.7),
           medianprops=dict(color='red', linewidth=2),
           flierprops=dict(marker='o', markerfacecolor='red', markersize=8, alpha=0.6))
ax.set_xlabel('Final Grade (G3)', fontsize=13)
ax.set_title('Figure 2: Box Plot of Final Grades (G3)', fontsize=14, fontweight='bold')
ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:
# D3: Box Plots for Key Variables
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for idx, col in enumerate(focus_cols):
    ax = axes[idx // 3][idx % 3]
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor=sns.color_palette('Set2')[idx], alpha=0.7),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='D', markerfacecolor='red', markersize=6))
    ax.set_title(col, fontsize=13, fontweight='bold')
fig.suptitle('Figure 3: Box Plots of Key Variables', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# D4: Studytime vs G3
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
st_means = df.groupby('studytime')['G3'].mean()
axes[0].bar(st_means.index, st_means.values, color=sns.color_palette('Set2', 4), edgecolor='black')
axes[0].set_xlabel('Study Time Category')
axes[0].set_ylabel('Mean Final Grade (G3)')
axes[0].set_title('Mean G3 by Study Time', fontweight='bold')
axes[0].set_xticks([1, 2, 3, 4])
axes[0].set_xticklabels(['<2 hrs', '2-5 hrs', '5-10 hrs', '>10 hrs'])

parts = axes[1].violinplot([df[df['studytime']==t]['G3'].values for t in sorted(df['studytime'].unique())],
                            positions=sorted(df['studytime'].unique()), showmeans=True, showmedians=True)
axes[1].set_xlabel('Study Time Category')
axes[1].set_ylabel('Final Grade (G3)')
axes[1].set_title('G3 Distribution by Study Time', fontweight='bold')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xticklabels(['<2 hrs', '2-5 hrs', '5-10 hrs', '>10 hrs'])
fig.suptitle('Figure 4: Study Time vs Final Grade', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# D5: Scatter Plot G1 vs G3
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(df['G1'], df['G3'], c=df['G2'], cmap='viridis', alpha=0.7, edgecolors='gray', s=60)
plt.colorbar(scatter, ax=ax, label='G2')
z = np.polyfit(df['G1'], df['G3'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['G1'].min(), df['G1'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', lw=2, label=f'Trend: G3 = {z[0]:.2f}·G1 + {z[1]:.2f}')
r, pval = stats.pearsonr(df['G1'], df['G3'])
ax.set_xlabel('First Period Grade (G1)')
ax.set_ylabel('Final Grade (G3)')
ax.set_title(f'Figure 5: G1 vs G3 (r = {r:.3f})', fontsize=14, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# D6: Correlation Heatmap
corr_matrix = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax, annot_kws={'size': 8})
ax.set_title('Figure 7: Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('
Correlations with G3 (sorted):')
print(corr_matrix['G3'].drop('G3').sort_values(ascending=False))


## Section E: Outlier Detection (IQR Method)

In [ ]:
outlier_cols = ['absences', 'G1', 'G2', 'G3', 'failures', 'age', 'studytime']
outlier_summary = []
for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary.append({'Variable': col, 'Q1': Q1, 'Q3': Q3, 'IQR': IQR,
                            'Lower Bound': lower, 'Upper Bound': upper,
                            'Outlier Count': len(outliers), 'Outlier %': f'{len(outliers)/len(df)*100:.1f}%'})
    print(f"{col}: Q1={Q1}, Q3={Q3}, IQR={IQR}, Bounds=[{lower:.1f}, {upper:.1f}], Outliers={len(outliers)}")

pd.DataFrame(outlier_summary)


In [ ]:
# Outlier visualization
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for idx, col in enumerate(outlier_cols):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    is_outlier = (df[col] < lower) | (df[col] > upper)
    axes[idx].scatter(range(len(df)), df[col], c=['red' if o else '#4C72B0' for o in is_outlier], alpha=0.5, s=20)
    axes[idx].axhline(upper, color='orange', ls='--', label=f'Upper={upper:.1f}')
    axes[idx].axhline(lower, color='orange', ls='--', label=f'Lower={lower:.1f}')
    axes[idx].set_title(f'{col} (outliers: {is_outlier.sum()})', fontweight='bold')
    axes[idx].legend(fontsize=8)
axes[-1].set_visible(False)
fig.suptitle('Figure 8: Outlier Detection (IQR Method)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


## Section G: MLP Classifier (Pass/Fail Prediction) – Optional Extension

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

# Binary target: Pass (G3 >= 10) / Fail (G3 < 10)
df['pass_fail'] = (df['G3'] >= 10).astype(int)
print(f"Pass/Fail distribution:")
print(df['pass_fail'].value_counts().rename({1: 'Pass', 0: 'Fail'}))
print(f"Pass rate: {df['pass_fail'].mean()*100:.1f}%")


In [ ]:
# Features (excluding G1, G2 to prevent target leakage)
num_features = ['age','Medu','Fedu','traveltime','studytime','failures','famrel','freetime','goout','Dalc','Walc','health','absences']
cat_features = ['school','sex','address','famsize','Pstatus','Mjob','Fjob','reason','guardian','schoolsup','famsup','paid','activities','nursery','higher','internet','romantic']

df_ml = df.copy()
for col in cat_features:
    df_ml[col] = LabelEncoder().fit_transform(df_ml[col])

X = df_ml[num_features + cat_features]
y = df_ml['pass_fail']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

mlp = MLPClassifier(hidden_layer_sizes=(64,32), activation='relu', solver='adam',
                    max_iter=500, random_state=42, early_stopping=True,
                    validation_fraction=0.15, n_iter_no_change=20, learning_rate_init=0.001)
mlp.fit(X_train_s, y_train)
y_pred = mlp.predict(X_test_s)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"
{classification_report(y_test, y_pred, target_names=['Fail','Pass'])}")

fig, ax = plt.subplots(figsize=(8,6))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['Fail','Pass']).plot(ax=ax, cmap='Blues')
ax.set_title('Confusion Matrix – MLP Classifier', fontweight='bold')
plt.show()


In [ ]:
# Training loss curve
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(mlp.loss_curve_, color='#4C72B0', lw=2, label='Training Loss')
if hasattr(mlp, 'validation_scores_'):
    ax2 = ax.twinx()
    ax2.plot(mlp.validation_scores_, color='#C44E52', lw=2, label='Validation Accuracy')
    ax2.set_ylabel('Validation Accuracy', color='#C44E52')
    ax2.legend(loc='center right')
ax.set_xlabel('Iteration')
ax.set_ylabel('Training Loss', color='#4C72B0')
ax.set_title('MLP Training Loss Curve', fontweight='bold')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()
